In [16]:
import pandas as pd
import streamlit as st
import numpy as np
import plotly.express as pe



In [17]:
pedidos = pd.read_csv("KORA_pedidos.csv",parse_dates=["fecha_pedido"])
clientes=pd.read_csv("KORA_clientes.csv",parse_dates=["fecha_registro",])
pedidos



,id_pedido,id_cliente,fecha_pedido,categoria,unidades,ingreso_neto_cop
0,P00001,C0892,2025-08-21,Hogar,1,113000
1,P00002,C1308,2025-08-22,Deporte,1,136000
2,P00003,C0213,2025-08-30,Cuidado personal,1,89000
3,P00004,C0012,2025-09-02,Deporte,2,283000
4,P00005,C0138,2025-09-03,Hogar,1,200000
...,...,...,...,...,...,...
2586,P02587,C1039,2026-10-31,Cuidado personal,2,104000
2587,P02588,C1081,2026-10-31,Tecnología,2,218000
2588,P02589,C1082,2026-10-31,Deporte,1,172000
2589,P02590,C1127,2026-10-31,Hogar,4,315000


In [18]:
clientes

,id_cliente,fecha_registro,ciudad,canal_adquisicion,preferencia_declarada
0,C0001,2026-06-19,Bogotá,Redes sociales,Cuidado personal
1,C0002,2026-05-25,Cali,Referidos,Papelería
2,C0003,2025-12-01,Medellín,Orgánico,Deporte
3,C0004,2026-09-09,Medellín,Referidos,Tecnología
4,C0005,2026-08-08,Bogotá,Redes sociales,Tecnología
...,...,...,...,...,...
1345,C1346,2026-05-12,Bogotá,Redes sociales,Hogar
1346,C1347,2025-12-10,Bogotá,Referidos,Tecnología
1347,C1348,2026-08-21,Medellín,Referidos,Papelería
1348,C1349,2025-10-16,Bogotá,Orgánico,Hogar


In [19]:
pedidos.dtypes

id_pedido                      str
id_cliente                     str
fecha_pedido        datetime64[us]
categoria                      str
unidades                     int64
ingreso_neto_cop             int64
dtype: object

In [20]:
compradores = pedidos.groupby("id_cliente").agg(n_pedidos=("id_pedido","count"))
compradores

,n_pedidos
id_cliente,
C0001,1
C0002,2
C0005,1
C0006,2
C0007,1
...,...
C1344,2
C1346,1
C1347,1


In [21]:
clientes.columns

Index(['id_cliente', 'fecha_registro', 'ciudad', 'canal_adquisicion',
       'preferencia_declarada'],
      dtype='str')

In [22]:
final = clientes.merge(compradores, how = "left", on = "id_cliente")
final["n_pedidos"] = final["n_pedidos"].fillna(0).astype(int)

In [23]:
final

,id_cliente,fecha_registro,ciudad,canal_adquisicion,preferencia_declarada,n_pedidos
0,C0001,2026-06-19,Bogotá,Redes sociales,Cuidado personal,1
1,C0002,2026-05-25,Cali,Referidos,Papelería,2
2,C0003,2025-12-01,Medellín,Orgánico,Deporte,0
3,C0004,2026-09-09,Medellín,Referidos,Tecnología,0
4,C0005,2026-08-08,Bogotá,Redes sociales,Tecnología,1
...,...,...,...,...,...,...
1345,C1346,2026-05-12,Bogotá,Redes sociales,Hogar,1
1346,C1347,2025-12-10,Bogotá,Referidos,Tecnología,1
1347,C1348,2026-08-21,Medellín,Referidos,Papelería,0
1348,C1349,2025-10-16,Bogotá,Orgánico,Hogar,5


### PUNTO 3

In [65]:
compradores_df = pedidos.groupby("id_cliente").agg(
    n_pedidos=("id_pedido", "count"),
    ingreso_total=("ingreso_neto_cop", "sum"),
    fecha_ultimo_pedido=("fecha_pedido", "max")
)
compradores_df["recencia"] = (pd.Timestamp.today() - compradores_df["fecha_ultimo_pedido"]).dt.days
compradores_df.head()

,n_pedidos,ingreso_total,fecha_ultimo_pedido,recencia
id_cliente,,,,
C0001,1,199000,2026-09-09,66
C0002,2,440000,2026-10-26,19
C0005,1,203000,2026-08-27,79
C0006,2,104000,2026-10-26,19
C0007,1,63000,2026-08-18,88


In [66]:
def segmento(n_pedidos, tiempo):
    if n_pedidos == 0:
        return "Sin Compra"
    elif n_pedidos == 1 and tiempo <= 90:
        return "Primerizo"
    elif n_pedidos == 2 and tiempo <= 90:
        return "En Desarrollo"
    elif n_pedidos >= 3 and tiempo <= 90:
        return "Fiel"
    else:
        return "En Pausa"

segmento(1, 30), segmento(0, None), segmento(4, 200)

('Primerizo', 'Sin Compra', 'En Pausa')

In [67]:
final = clientes.merge(compradores_df, how="left", on="id_cliente")
final["n_pedidos"] = final["n_pedidos"].fillna(0).astype(int)
final["ingreso_total"] = final["ingreso_total"].fillna(0)
final["segmento"] = final.apply(lambda fila: segmento(fila["n_pedidos"], fila["recencia"]), axis=1)
final["segmento"].value_counts()

segmento
En Pausa         341
Sin Compra       312
Fiel             279
En Desarrollo    213
Primerizo        205
Name: count, dtype: int64

In [68]:
filtro_canal = final.canal_adquisicion.unique()
filtro_ciudad = final.ciudad.unique()
filtro_segmento = final.segmento.unique()

final_FILTRADO = final[
    (final.canal_adquisicion.isin(filtro_canal)) &
    (final.ciudad.isin(filtro_ciudad)) &
    (final.segmento.isin(filtro_segmento))
]
final_FILTRADO.shape

(1350, 10)

In [69]:
compradores_sel = final_FILTRADO[final_FILTRADO["n_pedidos"] > 0]

por_canal = compradores_sel.groupby("canal_adquisicion").agg(
    compradores=("id_cliente", "count"),
    recompradores=("n_pedidos", lambda x: (x >= 2).sum()),
    una_compra=("n_pedidos", lambda x: (x == 1).sum())
).reset_index()

por_canal["tasa_recompra"] = por_canal["recompradores"] / por_canal["compradores"]
por_canal

,canal_adquisicion,compradores,recompradores,una_compra,tasa_recompra
0,Búsqueda,262,179,83,0.683206
1,Orgánico,196,148,48,0.755102
2,Redes sociales,364,217,147,0.596154
3,Referidos,216,142,74,0.657407


In [70]:


tabla_tasa = por_canal.sort_values("tasa_recompra", ascending=False)

fig_tasa = pe.bar(
    tabla_tasa,
    x="canal_adquisicion",
    y="tasa_recompra",
    text_auto=".1%",
    hover_data=["compradores", "recompradores"],
    labels={"canal_adquisicion": "Canal", "tasa_recompra": "Tasa de recompra",
            "compradores": "Compradores", "recompradores": "Con 2+ pedidos"},
    title="% de compradores con 2 o más pedidos, por canal"
)
fig_tasa.update_layout(yaxis_tickformat=".0%")
fig_tasa.show()

Usamos un gráfico de barras porque solo comparamos un dato, el porcentaje de recompra, entre cuatro canales. Con barras se ve rápido qué canal tiene la tasa más alta y cuál la más baja, y las ordenamos de mayor a menor para que sea más fácil leerlas. Al pasar el mouse sobre cada barra aparece cuántos compradores tiene ese canal, para saber si el porcentaje se basa en muchos o en pocos clientes.

### punto 4

In [74]:
compradores_sel["una_compra"] = compradores_sel["n_pedidos"] == 1

df_1 = (compradores_sel.groupby("canal_adquisicion", as_index=False)
          .agg(una_compra=("una_compra", "sum")))
df_1 = df_1.sort_values("una_compra", ascending=True)
df_1

,canal_adquisicion,una_compra
1,Orgánico,48
3,Referidos,74
0,Búsqueda,83
2,Redes sociales,147


In [76]:
fig4 = pe.bar(data_frame=df_1, x="una_compra", y="canal_adquisicion",
              orientation="h", text="una_compra")
fig4.update_layout(xaxis_title="Clientes con 1 pedido", yaxis_title="Canal",
                   title="Clientes con exactamente un pedido por canal")
fig4.show()